# 02 - Train and register the defect classifier

Build plane, `train` task of the `msd_training` job (serverless CPU).

* **Model**: torchvision MobileNetV3-Small pretrained on ImageNet, classifier head replaced with
  the 6 NEU classes. Grayscale images are replicated to 3 channels and resized to 224x224;
  augmentation: horizontal and vertical flips.
* **Training**: AdamW (lr 3e-4, weight decay 1e-4), cosine schedule, 8 epochs, batch 32; the
  checkpoint with the best validation accuracy is kept. About 5 minutes on serverless CPU.
  Moving to GPU only requires changing the task compute.
* **Packaging**: MLflow pyfunc. Input: DataFrame with column `image` (base64-encoded image
  bytes). Output: `predicted_class`, `confidence`, `probabilities` (JSON class -> probability).
  Signature and input example are logged; dependencies pin the CPU PyTorch build.
* **Serving-readiness gate**: the logged artifact is loaded and scored in an isolated `uv`
  environment (installs the declared requirements from scratch) before registration.
* **Registration**: new version of `<catalog>.<schema>.<model_name>` in Unity Catalog. The
  `champion` alias is set later by `04_evaluate` only if the test-accuracy gate passes.

In [0]:
%pip install -q torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cpu
%pip install -q uv
%restart_python

In [0]:
dbutils.widgets.text("catalog", "poc_metal_surface_defect_diagnostics")
dbutils.widgets.text("schema", "metal_surface_defects")
dbutils.widgets.text("model_name", "msd_defect_classifier")
dbutils.widgets.text("experiment_path", "")
dbutils.widgets.text("epochs", "8")
dbutils.widgets.text("seed", "42")

catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
registered_model_name = f"{catalog}.{schema}.{dbutils.widgets.get('model_name')}"
epochs = int(dbutils.widgets.get("epochs"))
seed = int(dbutils.widgets.get("seed"))

current_user = spark.sql("SELECT current_user()").first()[0]
experiment_path = dbutils.widgets.get("experiment_path") or f"/Users/{current_user}/msd_defect_classifier"

index_table = f"{catalog}.{schema}.image_index"
bronze_table = f"{catalog}.{schema}.image_bronze"

# Pinned versions (also declared as serving requirements below)
TORCH_VERSION, TORCHVISION_VERSION = "2.5.1", "0.20.1"
IMAGE_SIZE, BATCH_SIZE, LR, WEIGHT_DECAY = 224, 32, 3e-4, 1e-4

print(f"model: {registered_model_name}\nexperiment: {experiment_path}\nepochs: {epochs}")

In [0]:
pdf = spark.sql(f"""
    SELECT i.file_name, i.label, i.split, b.content
    FROM {index_table} i
    JOIN {bronze_table} b ON i.path = regexp_replace(b.path, '^dbfs:', '')
""").toPandas()

CLASSES = sorted(pdf["label"].unique())
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
split_counts = pdf.groupby("split").size().to_dict()
assert len(CLASSES) == 6 and all(split_counts.get(s, 0) > 0 for s in ("train", "val", "test")), split_counts
print(f"classes: {CLASSES}\nsplits: {split_counts}")

In [0]:
import io

import torch
import torchvision
from PIL import Image
from torchvision import transforms as T

torch.manual_seed(seed)

NORMALIZE = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([T.Grayscale(3), T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
                      T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.ToTensor(), NORMALIZE])
eval_tf = T.Compose([T.Grayscale(3), T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(), NORMALIZE])


class ImageDataset(torch.utils.data.Dataset):
    def __init__(self, df, transform):
        self.images = [Image.open(io.BytesIO(b)).convert("L") for b in df["content"]]
        self.labels = [CLASS_TO_IDX[l] for l in df["label"]]
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.transform(self.images[i]), self.labels[i]


train_loader = torch.utils.data.DataLoader(ImageDataset(pdf[pdf.split == "train"], train_tf), batch_size=BATCH_SIZE, shuffle=True)
val_loader = torch.utils.data.DataLoader(ImageDataset(pdf[pdf.split == "val"], eval_tf), batch_size=64)

In [0]:
import copy
import time

import mlflow

mlflow.set_experiment(experiment_path)


def build_model(num_classes, pretrained=True):
    weights = torchvision.models.MobileNet_V3_Small_Weights.IMAGENET1K_V1 if pretrained else None
    model = torchvision.models.mobilenet_v3_small(weights=weights)
    model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, num_classes)
    return model


def accuracy(model, loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item()
            total += len(y)
    return correct / total


model = build_model(len(CLASSES))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs * len(train_loader))
loss_fn = torch.nn.CrossEntropyLoss()

run = mlflow.start_run(run_name="mobilenet_v3_small_cpu")
mlflow.log_params({
    "backbone": "mobilenet_v3_small", "pretrained": "IMAGENET1K_V1", "epochs": epochs,
    "batch_size": BATCH_SIZE, "lr": LR, "weight_decay": WEIGHT_DECAY, "image_size": IMAGE_SIZE,
    "seed": seed, "classes": ",".join(CLASSES), "train_images": split_counts["train"],
    "val_images": split_counts["val"], "torch": torch.__version__,
})

best_acc, best_state, best_epoch = -1.0, None, -1
for epoch in range(epochs):
    start = time.time()
    model.train()
    running = 0.0
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
        scheduler.step()
        running += loss.item() * len(y)
    val_acc = accuracy(model, val_loader)
    train_loss = running / len(train_loader.dataset)
    mlflow.log_metrics({"train_loss": train_loss, "val_accuracy": val_acc}, step=epoch)
    if val_acc > best_acc:
        best_acc, best_state, best_epoch = val_acc, copy.deepcopy(model.state_dict()), epoch
    print(f"epoch {epoch}: loss {train_loss:.4f} val_acc {val_acc:.3f} ({time.time() - start:.0f}s)")

model.load_state_dict(best_state)
mlflow.log_metrics({"best_val_accuracy": best_acc, "best_epoch": best_epoch})
print(f"best val accuracy {best_acc:.3f} at epoch {best_epoch}")

In [0]:
import base64
import json
import os
import tempfile

import pandas as pd
from mlflow.models import infer_signature


class DefectClassifier(mlflow.pyfunc.PythonModel):
    """Input: DataFrame column `image` with base64-encoded image bytes (any PIL-readable format).
    Output: predicted_class, confidence, probabilities (JSON string class -> probability)."""

    def load_context(self, context):
        import torch
        import torchvision
        from torchvision import transforms as T

        with open(context.artifacts["classes"]) as f:
            self.classes = json.load(f)
        model = torchvision.models.mobilenet_v3_small(weights=None)
        model.classifier[3] = torch.nn.Linear(model.classifier[3].in_features, len(self.classes))
        model.load_state_dict(torch.load(context.artifacts["weights"], map_location="cpu", weights_only=True))
        self.model = model.eval()
        self.transform = T.Compose([
            T.Grayscale(3), T.Resize((224, 224)), T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ])

    def predict(self, context, model_input, params=None):
        import torch
        from PIL import Image

        images = [Image.open(io.BytesIO(base64.b64decode(s))).convert("L") for s in model_input["image"]]
        batch = torch.stack([self.transform(img) for img in images])
        with torch.no_grad():
            probs = torch.softmax(self.model(batch), dim=1).numpy()
        rows = []
        for p in probs:
            top = int(p.argmax())
            rows.append({
                "predicted_class": self.classes[top],
                "confidence": float(p[top]),
                "probabilities": json.dumps({c: round(float(v), 6) for c, v in zip(self.classes, p)}),
            })
        return pd.DataFrame(rows, columns=["predicted_class", "confidence", "probabilities"])


artifact_dir = tempfile.mkdtemp()
weights_path, classes_path = os.path.join(artifact_dir, "weights.pt"), os.path.join(artifact_dir, "classes.json")
torch.save(model.state_dict(), weights_path)
with open(classes_path, "w") as f:
    json.dump(CLASSES, f)

example_rows = pdf[pdf.split == "val"].groupby("label").head(1)
input_example = pd.DataFrame({"image": [base64.b64encode(b).decode() for b in example_rows["content"]]})

# Local smoke test of the wrapper before logging (signature inference uses its output)
wrapper = DefectClassifier()
wrapper.load_context(mlflow.pyfunc.PythonModelContext(artifacts={"weights": weights_path, "classes": classes_path}, model_config={}))
example_output = wrapper.predict(None, input_example)
print(pd.concat([example_rows["label"].reset_index(drop=True), example_output[["predicted_class", "confidence"]]], axis=1))

model_info = mlflow.pyfunc.log_model(
    name="model",
    python_model=DefectClassifier(),
    artifacts={"weights": weights_path, "classes": classes_path},
    signature=infer_signature(input_example, example_output),
    input_example=input_example,
    pip_requirements=[
        "--extra-index-url https://download.pytorch.org/whl/cpu",
        # No "+cpu" local tag: that build only exists for x86_64; without it the CPU index still
        # yields the CPU build on x86_64 and a plain wheel on aarch64 serverless machines.
        f"torch=={TORCH_VERSION}",
        f"torchvision=={TORCHVISION_VERSION}",
        "pillow",
        "pandas",
        "numpy",
        f"mlflow=={mlflow.__version__}",
    ],
)
print(f"logged: {model_info.model_uri}")

In [0]:
# Installs the declared pip requirements from scratch in a fresh uv environment, then loads and
# scores the logged artifact. Registration only happens if this succeeds.
logged = mlflow.pyfunc.load_model(model_info.model_uri)
assert logged.metadata.signature is not None, "logged model has no signature"
assert logged.input_example is not None, "logged model has no input example"

# One retry: package-index downloads (~200 MB for CPU torch) occasionally fail transiently,
# and failing here would otherwise retrain from scratch on task retry.
for attempt in (1, 2):
    try:
        mlflow.models.predict(
            model_uri=model_info.model_uri,
            input_data=logged.input_example,
            env_manager="uv",
        )
        break
    except Exception as e:
        if attempt == 2:
            raise
        print(f"isolated check attempt {attempt} failed ({type(e).__name__}); retrying once")
print("isolated serving-readiness check passed")

In [0]:
mlflow.set_registry_uri("databricks-uc")
registered_version = mlflow.register_model(
    model_uri=model_info.model_uri,
    name=registered_model_name,
    await_registration_for=300,
)
model_version = registered_version.version
mlflow.end_run()

from mlflow import MlflowClient

client = MlflowClient(registry_uri="databricks-uc")
client.update_registered_model(
    registered_model_name,
    description="Metal Surface Defect Diagnostics: MobileNetV3-Small classifier for the 6 NEU surface defect classes. "
                "Input: base64 image; output: predicted_class, confidence, probabilities (JSON).",
)

# Hand the exact version to the evaluate task
dbutils.jobs.taskValues.set(key="model_version", value=str(model_version))
print(f"registered {registered_model_name} version {model_version}")